# 04.2.2 - Supervised Learning: Klasifikasi SVM (K-Means QLDE)

## 1. Tujuan Analisis
Setelah menguji K-Means QLDE dengan *Decision Tree*, sekarang kita akan mengujinya menggunakan "mesin penebak" yang lebih tangguh, yaitu **Support Vector Machine (SVM)**. 

Tujuan utama dari *notebook* ini adalah melihat berapa **akurasi tertinggi** yang bisa dicapai model saat mencoba mengenali label pelanggan dari hasil algoritma QLDE. Apakah performanya akan lebih baik, sama, atau di bawah hasil K-Means standar?

## 2. Kapasitas Komputasi
Karena total dataset yang kita miliki saat ini relatif efisien (berada di kisaran 4.000-an baris). Seluruh data latih akan digunakan secara penuh agar model dapat mempelajari karakteristik setiap kelompok (*cluster*) hasil QLDE dengan maksimal.

In [ ]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. BACA DATA (Gunakan Pemenang Clustering: QLDE)
df = pd.read_csv('../data/Labeled/hasildata_kmeans-qlde.csv')

# Pisahkan Fitur (Var1 - Var11) dan Target (Cluster)
fitur = [f'Var{i}' for i in range(1, 12)]
X = df[fitur]
y = df['Cluster']

# Splitting Data: 80% Train untuk belajar, 20% Test untuk ujian
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Menampilkan jumlah data untuk memastikan seluruh data digunakan
print(f"Jumlah data latih (Train): {X_train.shape[0]} baris")
print(f"Jumlah data uji (Test): {X_test.shape[0]} baris")

Jumlah data latih (Train): 3468 baris
None
Jumlah data uji (Test): 867 baris
None


## 3. Scaling dan Training Model
Seperti biasa, sebelum memasukkan data ke dalam model SVM, kita wajib menyetarakan angka-angkanya (Scaling) menggunakan `StandardScaler` agar model tidak bias terhadap fitur yang kebetulan memiliki satuan angka besar (seperti nilai transaksi).

In [7]:
# 2. SCALING DATA
scaler = StandardScaler()

# Model scaler belajar dari seluruh data latih (Train) sekaligus menyetarakan angkanya
X_train_scaled = scaler.fit_transform(X_train)

# Data ujian (Test) juga disetarakan menggunakan parameter dari data train di atas
X_test_scaled = scaler.transform(X_test)

# 3. TRAINING MODEL SVM
# Menggunakan kernel 'rbf' untuk menangani pola data yang kompleks
model_svm = SVC(kernel='rbf', random_state=42)
model_svm.fit(X_train_scaled, y_train)

# Evaluasi model menggunakan data ujian
prediksi_svm = model_svm.predict(X_test_scaled)
print("=== CLASSIFICATION REPORT: SVM (QLDE) ===\n")
print(classification_report(y_test, prediksi_svm))

=== CLASSIFICATION REPORT: SVM (QLDE) ===

              precision    recall  f1-score   support

           0       0.88      0.93      0.91       150
           1       0.92      0.96      0.94       214
           2       0.89      0.95      0.92       125
           3       0.92      0.94      0.93        85
           4       0.86      0.93      0.90       199
           5       0.96      0.50      0.66        94

    accuracy                           0.90       867
   macro avg       0.91      0.87      0.88       867
weighted avg       0.90      0.90      0.89       867



### Analisis Performa (QLDE vs Standard)
Dari rapor klasifikasi di atas, akurasi SVM pada data QLDE berhasil mencapai angka **90%** (atau `0.90`). 

Jika kita bandingkan, akurasi umum ini **setara** dengan performa SVM pada dataset K-Means Standard (yang juga mencapai 90%). Namun, ada catatan khusus: model tampak cukup kesulitan dalam mengenali pelanggan di **Class 5** (tingkat *recall* hanya 50%). Ini menunjukkan bahwa meskipun metode QLDE mampu dipelajari dengan baik secara keseluruhan, sebaran pelanggan di dalam Class 5 cukup tumpang tindih dengan kelompok lain.

## 4. Ekspor Model dan Scaler
Kita menyimpan file mesin penebak dan alat *scaler*-nya dengan akhiran `_qlde` agar siap digunakan dan tidak menimpa file dari eksperimen sebelumnya.

In [ ]:
# 4. EXPORT SCALER & MODEL KE FOLDER 'models'
os.makedirs('../models', exist_ok=True)

# Simpan Scaler
joblib.dump(scaler, '../models/scaler_svm_qlde.pkl')

# Simpan Model SVM
joblib.dump(model_svm, '../models/model_svm_classification_qlde.pkl')

print("\n[SUCCESS] Scaler & Model SVM tanpa batasan sampel berhasil diekspor ke folder '../models/'!")


[SUCCESS] Scaler & Model SVM tanpa batasan sampel berhasil diekspor ke folder '../models/'!
